# Final Project. Relational Databases
## Task 1. Create Schema and Import Data

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas gdown

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

# Start PostgreSQL 16
pg = pgserver.get_server('/tmp/final_project_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

# Check PostgreSQL version
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 55.5 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
import gdown

# Download the dataset from the course Google Drive
file_id = "1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"
csv_path = "/content/infectious_cases.csv"

gdown.download(
    id=file_id,
    output=csv_path,
    quiet=False
)

# Preview the dataset
df = pd.read_csv(csv_path)

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

display(df.head())

Downloading...
From: https://drive.google.com/uc?id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl
To: /content/infectious_cases.csv
100%|██████████| 541k/541k [00:00<00:00, 89.5MB/s]

Rows: 10521
Columns: 12


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Create the pandemic schema and set the search path
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS pandemic"))
    conn.execute(text("SET search_path TO pandemic, public"))

print("Schema pandemic created successfully.")

Schema pandemic created successfully.


In [4]:
# Create the original infectious cases table
create_table_sql = """
DROP TABLE IF EXISTS pandemic.infectious_cases CASCADE;

CREATE TABLE pandemic.infectious_cases (
    "Entity" TEXT,
    "Code" TEXT,
    "Year" TEXT,
    "Number_yaws" TEXT,
    "polio_cases" TEXT,
    "cases_guinea_worm" TEXT,
    "Number_rabies" TEXT,
    "Number_malaria" TEXT,
    "Number_hiv" TEXT,
    "Number_tuberculosis" TEXT,
    "Number_smallpox" TEXT,
    "Number_cholera_cases" TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(create_table_sql))

print("Original table created successfully.")

Original table created successfully.


In [5]:
# Import the CSV into PostgreSQL using COPY FROM STDIN
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert("""
                COPY pandemic.infectious_cases
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ',',
                    NULL ''
                )
            """, file)

    raw_conn.commit()
finally:
    raw_conn.close()

print("CSV imported successfully.")

CSV imported successfully.


In [6]:
# Verify imported records
summary_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT "Entity") AS unique_entities,
        MIN("Year"::INTEGER) AS first_year,
        MAX("Year"::INTEGER) AS last_year
    FROM pandemic.infectious_cases
"""), engine)

display(summary_df)

assert summary_df.iloc[0]["total_rows"] == 10521
assert summary_df.iloc[0]["unique_entities"] == 245

print("Data import verification passed.")

,total_rows,unique_entities,first_year,last_year
0,10521,245,1920,2023


Data import verification passed.


In [7]:
# Preview the imported data
data_df = pd.read_sql(text("""
    SELECT *
    FROM pandemic.infectious_cases
    LIMIT 10
"""), engine)

display(data_df)

,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867,0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901,0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396,0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149,0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385,0,None,None,None,None,None,None


In [8]:
# Check missing values in key columns
missing_df = pd.read_sql(text("""
    SELECT
        COUNT(*) FILTER (
            WHERE "Code" IS NULL
        ) AS missing_code,

        COUNT(*) FILTER (
            WHERE "Number_rabies" IS NULL
        ) AS missing_rabies,

        COUNT(*) FILTER (
            WHERE "Year" IS NULL
        ) AS missing_year

    FROM pandemic.infectious_cases
"""), engine)

display(missing_df)

,missing_code,missing_rabies,missing_year
0,1128,3741,0


### Результат

Було створено схему `pandemic` та таблицю `infectious_cases`, у яку завантажено дані з CSV-файлу.

Для імпорту використано команду `COPY FROM STDIN`.

Після завантаження було перевірено кількість записів, унікальних сутностей та діапазон років. Усього отримано 10 521 запис і 245 унікальних сутностей.

Також було перевірено пропущені значення. У колонці `Code` знайдено 1128 пропусків, а в `Number_rabies` - 3741. Ці значення залишено як `NULL`, щоб не замінювати їх штучними даними.

Початкова таблиця готова до нормалізації.

## Task 2. Normalization to Third Normal Form (3NF)

In [9]:
# Create a table for unique entities
entities_sql = """
DROP TABLE IF EXISTS pandemic.infectious_cases_normalized CASCADE;
DROP TABLE IF EXISTS pandemic.entities CASCADE;

CREATE TABLE pandemic.entities (
    entity_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_name TEXT NOT NULL,
    entity_code TEXT,

    CONSTRAINT uq_entity_name_code
        UNIQUE NULLS NOT DISTINCT (entity_name, entity_code)
);
"""

with engine.begin() as conn:
    conn.execute(text(entities_sql))

print("Entities table created successfully.")

Entities table created successfully.


In [10]:
# Insert unique combinations of entity and code
insert_entities_sql = """
INSERT INTO pandemic.entities (
    entity_name,
    entity_code
)
SELECT DISTINCT
    "Entity",
    "Code"
FROM pandemic.infectious_cases
ORDER BY "Entity", "Code";
"""

with engine.begin() as conn:
    conn.execute(text(insert_entities_sql))

print("Unique entities inserted successfully.")

Unique entities inserted successfully.


In [11]:
# Verify unique entities
entities_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_entities,
        COUNT(*) FILTER (
            WHERE entity_code IS NULL
        ) AS entities_without_code
    FROM pandemic.entities
"""), engine)

display(entities_df)

assert entities_df.iloc[0]["total_entities"] == 245

print("Entities verification passed.")

,total_entities,entities_without_code
0,245,34


Entities verification passed.


In [12]:
# Preview the entities table
df = pd.read_sql(text("""
    SELECT
        entity_id,
        entity_name,
        entity_code
    FROM pandemic.entities
    ORDER BY entity_id
    LIMIT 10
"""), engine)

display(df)

,entity_id,entity_name,entity_code
0,1,Afghanistan,AFG
1,2,Africa,None
2,3,African Region (WHO),None
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,None
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


In [13]:
# Create the normalized infectious cases table
normalized_sql = """
CREATE TABLE pandemic.infectious_cases_normalized (
    case_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    entity_id INTEGER NOT NULL
        REFERENCES pandemic.entities(entity_id),

    year INTEGER NOT NULL
        CHECK (year BETWEEN 1900 AND 2100),

    number_yaws NUMERIC,
    polio_cases NUMERIC,
    cases_guinea_worm NUMERIC,
    number_rabies NUMERIC,
    number_malaria NUMERIC,
    number_hiv NUMERIC,
    number_tuberculosis NUMERIC,
    number_smallpox NUMERIC,
    number_cholera_cases NUMERIC,

    UNIQUE (entity_id, year)
);

CREATE INDEX idx_infectious_cases_entity
    ON pandemic.infectious_cases_normalized(entity_id);
"""

with engine.begin() as conn:
    conn.execute(text(normalized_sql))

print("Normalized disease statistics table created successfully.")

Normalized disease statistics table created successfully.


In [14]:
# Transfer disease statistics to the normalized table
insert_normalized_sql = """
INSERT INTO pandemic.infectious_cases_normalized (
    entity_id,
    year,
    number_yaws,
    polio_cases,
    cases_guinea_worm,
    number_rabies,
    number_malaria,
    number_hiv,
    number_tuberculosis,
    number_smallpox,
    number_cholera_cases
)
SELECT
    e.entity_id,
    NULLIF(TRIM(i."Year"), '')::INTEGER,
    NULLIF(TRIM(i."Number_yaws"), '')::NUMERIC,
    NULLIF(TRIM(i."polio_cases"), '')::NUMERIC,
    NULLIF(TRIM(i."cases_guinea_worm"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_rabies"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_malaria"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_hiv"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_tuberculosis"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_smallpox"), '')::NUMERIC,
    NULLIF(TRIM(i."Number_cholera_cases"), '')::NUMERIC

FROM pandemic.infectious_cases AS i

INNER JOIN pandemic.entities AS e
    ON i."Entity" = e.entity_name
   AND i."Code" IS NOT DISTINCT FROM e.entity_code;
"""

with engine.begin() as conn:
    conn.execute(text(insert_normalized_sql))

print("Disease statistics transferred successfully.")

Disease statistics transferred successfully.


In [15]:
# Compare original and normalized row counts
validation_df = pd.read_sql(text("""
    SELECT
        (SELECT COUNT(*)
         FROM pandemic.infectious_cases) AS original_rows,

        (SELECT COUNT(*)
         FROM pandemic.infectious_cases_normalized) AS normalized_rows,

        (SELECT COUNT(*)
         FROM pandemic.entities) AS entities
"""), engine)

display(validation_df)

assert validation_df.iloc[0]["original_rows"] == 10521
assert validation_df.iloc[0]["normalized_rows"] == 10521
assert validation_df.iloc[0]["entities"] == 245

print("Normalization verification passed.")

,original_rows,normalized_rows,entities
0,10521,10521,245


Normalization verification passed.


In [16]:
# Verify all normalized records reference existing entities
relationships_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_cases,
        COUNT(*) FILTER (
            WHERE e.entity_id IS NULL
        ) AS missing_entities
    FROM pandemic.infectious_cases_normalized AS c
    LEFT JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id
"""), engine)

display(relationships_df)

assert relationships_df.iloc[0]["missing_entities"] == 0

print("Foreign key verification passed.")

,total_cases,missing_entities
0,10521,0


Foreign key verification passed.


In [17]:
# Preview normalized data with country names
df = pd.read_sql(text("""
    SELECT
        e.entity_id,
        e.entity_name,
        e.entity_code,
        c.year,
        c.number_rabies,
        c.polio_cases,
        c.number_malaria
    FROM pandemic.infectious_cases_normalized AS c
    INNER JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id
    ORDER BY e.entity_name, c.year
    LIMIT 15
"""), engine)

display(df)

,entity_id,entity_name,entity_code,year,number_rabies,polio_cases,number_malaria
0,1,Afghanistan,AFG,1960,NaN,NaN,NaN
1,1,Afghanistan,AFG,1965,NaN,NaN,NaN
2,1,Afghanistan,AFG,1980,NaN,6160.0,NaN
3,1,Afghanistan,AFG,1981,NaN,5859.0,NaN
4,1,Afghanistan,AFG,1982,NaN,9730.0,NaN
5,1,Afghanistan,AFG,1983,NaN,13937.0,NaN
6,1,Afghanistan,AFG,1984,NaN,3864.0,NaN
7,1,Afghanistan,AFG,1985,NaN,13867.0,NaN
8,1,Afghanistan,AFG,1986,NaN,12901.0,NaN
9,1,Afghanistan,AFG,1987,NaN,4396.0,NaN


### Результат нормалізації

Було створено дві таблиці: `entities` та `infectious_cases_normalized`.

Перша містить 245 унікальних комбінацій країн або регіонів та їхніх кодів. Друга містить 10 521 запис із роками та показниками захворювань.

Для зв'язку між таблицями використано `entity_id`. Завдяки цьому назви країн і коди більше не повторюються в кожному записі статистики.

У таблицях додано первинні та зовнішні ключі, а також обмеження для перевірки даних.

Після нормалізації було перевірено кількість рядків і правильність зв'язків. Кількість записів залишилася такою самою, тобто дані не були втрачені.

Отримана структура відповідає 3НФ: інформація про країни зберігається окремо, а таблиця статистики містить показники, пов'язані з конкретною країною та роком.

## Task 3. Аналіз даних - Number_rabies

In [18]:
# Calculate rabies statistics for each entity
rabies_stats_df = pd.read_sql(text("""
    SELECT
        e.entity_id,
        e.entity_name,
        e.entity_code,

        ROUND(AVG(c.number_rabies), 2) AS avg_rabies,
        MIN(c.number_rabies) AS min_rabies,
        MAX(c.number_rabies) AS max_rabies,
        ROUND(SUM(c.number_rabies), 2) AS total_rabies

    FROM pandemic.infectious_cases_normalized AS c

    INNER JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id

    WHERE c.number_rabies IS NOT NULL

    GROUP BY
        e.entity_id,
        e.entity_name,
        e.entity_code

    ORDER BY avg_rabies DESC, e.entity_name
    LIMIT 10
"""), engine)

display(rabies_stats_df)

,entity_id,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,total_rabies
0,241,World,OWID_WRL,20192.37,14075.5080,24744.6580,605771.11
1,120,Lower Middle Income (WB),None,15193.96,10202.5280,19182.7950,455818.78
2,202,South Asia (WB),None,11729.89,7271.2754,15361.8780,351896.66
3,204,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.9580,342729.81
4,77,G20,None,10189.05,6339.0796,13164.8810,305671.38
5,97,India,IND,8599.17,5425.8726,11121.1420,257975.20
6,209,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.9770,142448.04
7,3,African Region (WHO),None,4649.70,4283.0730,4917.7840,139491.07
8,121,Low Income (WB),None,3568.11,3014.6170,3873.5330,107043.16
9,58,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.83


In [19]:
# Verify the aggregation result
assert len(rabies_stats_df) == 10
assert rabies_stats_df["avg_rabies"].notna().all()
assert rabies_stats_df["avg_rabies"].is_monotonic_decreasing

print("Rabies statistics verification passed.")

Rabies statistics verification passed.


In [20]:
# Check the number of available rabies observations
coverage_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(number_rabies) AS records_with_rabies,
        COUNT(*) - COUNT(number_rabies) AS missing_rabies,

        COUNT(DISTINCT entity_id) FILTER (
            WHERE number_rabies IS NOT NULL
        ) AS entities_with_rabies

    FROM pandemic.infectious_cases_normalized
"""), engine)

display(coverage_df)

,total_records,records_with_rabies,missing_rabies,entities_with_rabies
0,10521,6780,3741,226


### Результат

За допомогою `GROUP BY` та агрегатних функцій було розраховано статистику показника `Number_rabies` для різних країн і регіонів.

Для кожної сутності отримано середнє, мінімальне, максимальне значення та суму за всі доступні роки.

Результати відсортовано за середнім значенням у порядку спадання та обмежено першими 10 записами.

Під час розрахунків пропущені значення `NULL` не враховувалися, оскільки вони означають відсутність даних, а не нульову кількість захворювань.

Отримані результати дозволяють порівняти статистику захворюваності між різними країнами та регіонами за доступними спостереженнями.

## Task 4. Робота з датами та різницею в роках

In [21]:
# Calculate the difference in full years using built-in SQL functions
date_diff_df = pd.read_sql(text("""
    SELECT
        e.entity_name,
        c.year,

        MAKE_DATE(c.year, 1, 1) AS year_start_date,

        CURRENT_DATE AS current_date,

        EXTRACT(
            YEAR FROM AGE(
                CURRENT_DATE,
                MAKE_DATE(c.year, 1, 1)
            )
        )::INTEGER AS years_difference

    FROM pandemic.infectious_cases_normalized AS c

    INNER JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id

    ORDER BY e.entity_name, c.year
    LIMIT 20
"""), engine)

display(date_diff_df)

,entity_name,year,year_start_date,current_date,years_difference
0,Afghanistan,1960,1960-01-01,2026-10-18,66
1,Afghanistan,1965,1965-01-01,2026-10-18,61
2,Afghanistan,1980,1980-01-01,2026-10-18,46
3,Afghanistan,1981,1981-01-01,2026-10-18,45
4,Afghanistan,1982,1982-01-01,2026-10-18,44
5,Afghanistan,1983,1983-01-01,2026-10-18,43
6,Afghanistan,1984,1984-01-01,2026-10-18,42
7,Afghanistan,1985,1985-01-01,2026-10-18,41
8,Afghanistan,1986,1986-01-01,2026-10-18,40
9,Afghanistan,1987,1987-01-01,2026-10-18,39


In [22]:
# Verify the date calculation for the year 1996
test_date_df = pd.read_sql(text("""
    SELECT
        1996 AS year,
        MAKE_DATE(1996, 1, 1) AS year_start_date,
        CURRENT_DATE AS current_date,
        EXTRACT(
            YEAR FROM AGE(
                CURRENT_DATE,
                MAKE_DATE(1996, 1, 1)
            )
        )::INTEGER AS years_difference
"""), engine)

display(test_date_df)

,year,year_start_date,current_date,years_difference
0,1996,1996-01-01,2026-10-18,30


In [23]:
# Verify generated dates for all normalized records
validation_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(*) FILTER (
            WHERE EXTRACT(
                MONTH FROM MAKE_DATE(year, 1, 1)
            ) <> 1
            OR EXTRACT(
                DAY FROM MAKE_DATE(year, 1, 1)
            ) <> 1
        ) AS invalid_dates

    FROM pandemic.infectious_cases_normalized
"""), engine)

display(validation_df)

assert validation_df.iloc[0]["invalid_dates"] == 0

print("Date verification passed.")

,total_records,invalid_dates
0,10521,0


Date verification passed.


### Результат

За допомогою функції `MAKE_DATE()` було створено дату першого січня для кожного року з датасету.

Функція `CURRENT_DATE` використовується для отримання поточної дати, а `AGE()` разом з `EXTRACT()` - для розрахунку різниці в повних роках.

Наприклад, для 1996 року формується дата `1996-01-01`, а різниця з поточною датою у 2026 році становить 30 років.

Додатково було перевірено правильність створених дат. Усі дати відповідають першому січня потрібного року.

Такий підхід дозволяє виконувати обчислення з датами безпосередньо в SQL, без додаткової обробки в Python.

## Task 5. Custom SQL Functions

In [24]:
# Create a SQL function to calculate full years since January 1
years_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.years_since(input_year INTEGER)
RETURNS INTEGER
LANGUAGE SQL
STABLE
RETURNS NULL ON NULL INPUT
AS $$
    SELECT EXTRACT(
        YEAR FROM AGE(
            CURRENT_DATE,
            MAKE_DATE(input_year, 1, 1)
        )
    )::INTEGER;
$$;
"""

with engine.begin() as conn:
    conn.execute(text(years_function_sql))

print("Function pandemic.years_since() created successfully.")

Function pandemic.years_since() created successfully.


In [25]:
# Test years_since() with the year from the assignment
years_test_df = pd.read_sql(text("""
    SELECT
        1996 AS input_year,
        MAKE_DATE(1996, 1, 1) AS year_start_date,
        CURRENT_DATE AS current_date,
        pandemic.years_since(1996) AS years_difference
"""), engine)

display(years_test_df)

,input_year,year_start_date,current_date,years_difference
0,1996,1996-01-01,2026-10-18,30


In [26]:
# Use the custom function with normalized disease data
years_data_df = pd.read_sql(text("""
    SELECT
        e.entity_name,
        c.year,
        MAKE_DATE(c.year, 1, 1) AS year_start_date,
        CURRENT_DATE AS current_date,
        pandemic.years_since(c.year) AS years_difference

    FROM pandemic.infectious_cases_normalized AS c
    JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id

    ORDER BY e.entity_name, c.year
    LIMIT 15
"""), engine)

display(years_data_df)

,entity_name,year,year_start_date,current_date,years_difference
0,Afghanistan,1960,1960-01-01,2026-10-18,66
1,Afghanistan,1965,1965-01-01,2026-10-18,61
2,Afghanistan,1980,1980-01-01,2026-10-18,46
3,Afghanistan,1981,1981-01-01,2026-10-18,45
4,Afghanistan,1982,1982-01-01,2026-10-18,44
5,Afghanistan,1983,1983-01-01,2026-10-18,43
6,Afghanistan,1984,1984-01-01,2026-10-18,42
7,Afghanistan,1985,1985-01-01,2026-10-18,41
8,Afghanistan,1986,1986-01-01,2026-10-18,40
9,Afghanistan,1987,1987-01-01,2026-10-18,39


In [27]:
# Verify that the custom function matches the built-in calculation
years_validation_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE pandemic.years_since(year)
                IS DISTINCT FROM
                EXTRACT(
                    YEAR FROM AGE(
                        CURRENT_DATE,
                        MAKE_DATE(year, 1, 1)
                    )
                )::INTEGER
        ) AS mismatches
    FROM pandemic.infectious_cases_normalized
"""), engine)

display(years_validation_df)

assert years_validation_df.iloc[0]["mismatches"] == 0

print("years_since() verification passed.")

,total_records,mismatches
0,10521,0


years_since() verification passed.


In [28]:
# Create a SQL function for average disease cases per period
average_function_sql = """
CREATE OR REPLACE FUNCTION pandemic.average_cases(
    cases_per_year NUMERIC,
    period_divisor NUMERIC
)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
RETURNS NULL ON NULL INPUT
AS $$
    SELECT cases_per_year / NULLIF(period_divisor, 0);
$$;
"""

with engine.begin() as conn:
    conn.execute(text(average_function_sql))

print("Function pandemic.average_cases() created successfully.")

Function pandemic.average_cases() created successfully.


In [29]:
# Test the function with monthly, quarterly, and half-year periods
average_test_df = pd.read_sql(text("""
    SELECT
        120 AS annual_cases,
        pandemic.average_cases(120, 12) AS monthly_average,
        pandemic.average_cases(120, 4) AS quarterly_average,
        pandemic.average_cases(120, 2) AS half_year_average,
        pandemic.average_cases(120, 0) AS zero_divisor_result
"""), engine)

display(average_test_df)

,annual_cases,monthly_average,quarterly_average,half_year_average,zero_divisor_result
0,120,10.0,30.0,60.0,None


In [30]:
# Calculate monthly, quarterly, and half-year rabies averages
cases_data_df = pd.read_sql(text("""
    SELECT
        e.entity_name,
        c.year,
        c.number_rabies AS annual_rabies,

        ROUND(
            pandemic.average_cases(c.number_rabies, 12), 2
        ) AS monthly_average,

        ROUND(
            pandemic.average_cases(c.number_rabies, 4), 2
        ) AS quarterly_average,

        ROUND(
            pandemic.average_cases(c.number_rabies, 2), 2
        ) AS half_year_average

    FROM pandemic.infectious_cases_normalized AS c
    JOIN pandemic.entities AS e
        ON c.entity_id = e.entity_id

    WHERE c.number_rabies IS NOT NULL
      AND c.number_rabies > 0

    ORDER BY c.number_rabies DESC, e.entity_name
    LIMIT 15
"""), engine)

display(cases_data_df)

,entity_name,year,annual_rabies,monthly_average,quarterly_average,half_year_average
0,World,1990,24744.658,2062.05,6186.16,12372.33
1,World,1991,24559.182,2046.60,6139.80,12279.59
2,World,1992,24337.137,2028.09,6084.28,12168.57
3,World,1993,24127.453,2010.62,6031.86,12063.73
4,World,1994,23972.550,1997.71,5993.14,11986.28
5,World,1995,23905.242,1992.10,5976.31,11952.62
6,World,1996,23831.625,1985.97,5957.91,11915.81
7,World,1997,23652.562,1971.05,5913.14,11826.28
8,World,1998,23415.790,1951.32,5853.95,11707.90
9,World,1999,23185.303,1932.11,5796.33,11592.65


In [31]:
# Verify both custom SQL functions
function_validation_df = pd.read_sql(text("""
    SELECT
        pandemic.years_since(1996) =
            EXTRACT(
                YEAR FROM AGE(
                    CURRENT_DATE,
                    MAKE_DATE(1996, 1, 1)
                )
            )::INTEGER AS years_function_ok,

        pandemic.average_cases(120, 12) = 10 AS monthly_ok,
        pandemic.average_cases(120, 4) = 30 AS quarterly_ok,
        pandemic.average_cases(120, 2) = 60 AS half_year_ok,

        pandemic.average_cases(120, 0) IS NULL AS zero_divisor_ok
"""), engine)

display(function_validation_df)

assert function_validation_df.iloc[0].all()

print("All custom SQL function checks passed.")

,years_function_ok,monthly_ok,quarterly_ok,half_year_ok,zero_divisor_ok
0,True,True,True,True,True


All custom SQL function checks passed.


### Результат

Було створено дві власні SQL-функції.

Функція `years_since()` приймає рік і повертає кількість повних років між першим січня цього року та поточною датою. Її результат було порівняно з обчисленням через вбудовані функції PostgreSQL.

Друга функція `average_cases()` розраховує середню кількість захворювань за заданий період. Для перевірки використано дільники 12, 4 та 2, які відповідають місяцю, кварталу та півріччю.

Наприклад, для 120 випадків на рік було отримано 10 випадків на місяць, 30 на квартал і 60 на півріччя.

Також додано перевірку ділення на нуль. У такому випадку функція повертає `NULL`.

Обидві функції було використано у SQL-запитах до нормалізованої таблиці. Перевірки показали, що результати обчислень відповідають очікуваним значенням.